# 🌡️ Reto Semana 8: MeteoSense Analytics

## Sistema de Análisis de Datos Meteorológicos con NumPy

---

### 📋 Contexto del Proyecto

**MeteoSense** es una startup que instala redes de sensores meteorológicos en ciudades de México para monitorear condiciones ambientales. Has sido contratado como **Analista de Datos Junior** para procesar las mediciones de su red de sensores en la Ciudad de México.

```
╔══════════════════════════════════════════════════════════════════╗
║                    METEOSENSE ANALYTICS                         ║
║                 Red de Monitoreo CDMX 2024                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║    📍 Sensores Activos: 5 estaciones                            ║
║    📊 Mediciones: Temperatura, Humedad, CO2                     ║
║    ⏱️  Frecuencia: Lecturas cada hora (24/día)                  ║
║    📅 Período: 7 días de monitoreo                              ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

### 🎯 Objetivos de Aprendizaje

En este reto aplicarás:
- Creación y manipulación de arrays NumPy
- Indexación y slicing para acceso a datos
- Operaciones vectorizadas para cálculos eficientes
- Funciones estadísticas de NumPy
- Manejo de valores faltantes (NaN)
- Broadcasting para operaciones entre arrays

---

## 📦 Configuración Inicial

Ejecuta esta celda para cargar NumPy y preparar el entorno.

In [2]:
import numpy as np

# Configuración para reproducibilidad
np.random.seed(42)

print("✅ NumPy cargado correctamente")
print(f"   Versión: {np.__version__}")

✅ NumPy cargado correctamente
   Versión: 2.4.6


---

## 📊 Datos del Reto

### Estructura de los Datos

```
ESTACIONES DE MONITOREO CDMX
════════════════════════════════════════════

Índice │ Estación          │ Zona
───────┼───────────────────┼──────────────
   0   │ Coyoacán          │ Sur
   1   │ Azcapotzalco      │ Norte
   2   │ Xochimilco        │ Sur
   3   │ Tlalpan           │ Sur
   4   │ Miguel Hidalgo    │ Centro-Oeste

════════════════════════════════════════════

VARIABLES MONITOREADAS
────────────────────────────────────────────
• Temperatura (°C): Rango típico 10-35°C
• Humedad Relativa (%): Rango 20-90%
• CO2 (ppm): Rango típico 350-500 ppm
────────────────────────────────────────────
```

Ejecuta la siguiente celda para generar los datos de los sensores.

In [3]:
# ═══════════════════════════════════════════════════════════════════
#                    GENERACIÓN DE DATOS DE SENSORES
# ═══════════════════════════════════════════════════════════════════

# Nombres de estaciones
estaciones = ['Coyoacán', 'Azcapotzalco', 'Xochimilco', 'Tlalpan', 'Miguel Hidalgo']
n_estaciones = len(estaciones)
n_dias = 7
n_horas = 24

# ─────────────────────────────────────────────────────────────────────
# TEMPERATURA (°C)
# Array 3D: (estaciones, días, horas)
# ─────────────────────────────────────────────────────────────────────
# Base de temperatura por estación (algunas zonas más cálidas)
temp_base = np.array([22, 24, 20, 19, 23])  # Temperatura base por estación

# Variación diaria (ciclo día/noche)
hora_del_dia = np.arange(24)
variacion_diaria = 5 * np.sin((hora_del_dia - 6) * np.pi / 12)  # Máx a las 14h, mín a las 6h

# Generar datos de temperatura
temperatura = np.zeros((n_estaciones, n_dias, n_horas))
for i in range(n_estaciones):
    for d in range(n_dias):
        temperatura[i, d, :] = temp_base[i] + variacion_diaria + np.random.normal(0, 1.5, n_horas)

# Introducir algunos valores faltantes (sensores desconectados)
temperatura[1, 2, 10:14] = np.nan  # Azcapotzalco, día 3, horas 10-13
temperatura[3, 5, 0:3] = np.nan    # Tlalpan, día 6, horas 0-2

# ─────────────────────────────────────────────────────────────────────
# HUMEDAD RELATIVA (%)
# Array 3D: (estaciones, días, horas)
# ─────────────────────────────────────────────────────────────────────
humedad_base = np.array([55, 45, 70, 65, 50])  # Xochimilco y Tlalpan más húmedos

# Variación inversa a temperatura (más húmedo en la noche)
variacion_humedad = -15 * np.sin((hora_del_dia - 6) * np.pi / 12)

humedad = np.zeros((n_estaciones, n_dias, n_horas))
for i in range(n_estaciones):
    for d in range(n_dias):
        humedad[i, d, :] = humedad_base[i] + variacion_humedad + np.random.normal(0, 5, n_horas)

# Asegurar rango válido [20, 95]
humedad = np.clip(humedad, 20, 95)

# Valores faltantes
humedad[0, 4, 15:18] = np.nan  # Coyoacán, día 5, horas 15-17

# ─────────────────────────────────────────────────────────────────────
# NIVELES DE CO2 (ppm)
# Array 3D: (estaciones, días, horas)
# ─────────────────────────────────────────────────────────────────────
co2_base = np.array([380, 420, 360, 350, 410])  # Zonas con más tráfico tienen más CO2

# Patrón de hora pico (más CO2 en horas de tráfico)
patron_trafico = np.zeros(24)
patron_trafico[7:10] = 30   # Hora pico mañana
patron_trafico[17:20] = 40  # Hora pico tarde
patron_trafico[12:14] = 15  # Mediodía

co2 = np.zeros((n_estaciones, n_dias, n_horas))
for i in range(n_estaciones):
    for d in range(n_dias):
        co2[i, d, :] = co2_base[i] + patron_trafico + np.random.normal(0, 10, n_horas)

# Día de contingencia (día 4) - CO2 elevado
co2[:, 3, :] *= 1.15

# Valores faltantes
co2[2, 1, 5:8] = np.nan  # Xochimilco, día 2, horas 5-7

# ─────────────────────────────────────────────────────────────────────
# ARRAY 2D SIMPLIFICADO: Promedios diarios por estación
# ─────────────────────────────────────────────────────────────────────
# Para ejercicios más simples
temp_promedio_diario = np.nanmean(temperatura, axis=2)  # Shape: (5, 7)
humedad_promedio_diario = np.nanmean(humedad, axis=2)
co2_promedio_diario = np.nanmean(co2, axis=2)

print("╔══════════════════════════════════════════════════════════════╗")
print("║              DATOS GENERADOS EXITOSAMENTE                    ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  🌡️  temperatura     : shape {temperatura.shape}               ║")
print(f"║  💧 humedad         : shape {humedad.shape}               ║")
print(f"║  🏭 co2             : shape {co2.shape}               ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  📊 temp_promedio_diario    : shape {temp_promedio_diario.shape}            ║")
print(f"║  📊 humedad_promedio_diario : shape {humedad_promedio_diario.shape}            ║")
print(f"║  📊 co2_promedio_diario     : shape {co2_promedio_diario.shape}            ║")
print("╚══════════════════════════════════════════════════════════════╝")
print("\n📍 Estaciones:", estaciones)

╔══════════════════════════════════════════════════════════════╗
║              DATOS GENERADOS EXITOSAMENTE                    ║
╠══════════════════════════════════════════════════════════════╣
║  🌡️  temperatura     : shape (5, 7, 24)               ║
║  💧 humedad         : shape (5, 7, 24)               ║
║  🏭 co2             : shape (5, 7, 24)               ║
╠══════════════════════════════════════════════════════════════╣
║  📊 temp_promedio_diario    : shape (5, 7)            ║
║  📊 humedad_promedio_diario : shape (5, 7)            ║
║  📊 co2_promedio_diario     : shape (5, 7)            ║
╚══════════════════════════════════════════════════════════════╝

📍 Estaciones: ['Coyoacán', 'Azcapotzalco', 'Xochimilco', 'Tlalpan', 'Miguel Hidalgo']


---

## 🏋️ PARTE 1: Exploración de Arrays (25 puntos)

### Ejercicio 1.1: Inspección de Datos (5 puntos)

Completa el código para mostrar las propiedades del array `temperatura`.

In [5]:
# ═══════════════════════════════════════════════════════════════════
#                    EJERCICIO 1.1: INSPECCIÓN
# ═══════════════════════════════════════════════════════════════════

# TODO: Completa las siguientes líneas

# 1. Número de dimensiones del array temperatura
n_dimensiones = temperatura.ndim

# 2. Forma (shape) del array
forma = temperatura.shape

# 3. Número total de elementos
total_elementos = temperatura.size

# 4. Tipo de datos
tipo_datos = temperatura.dtype

# 5. Tamaño en memoria (bytes)
memoria_bytes = temperatura.nbytes

# Mostrar resultados
print("📊 PROPIEDADES DEL ARRAY TEMPERATURA")
print("─" * 40)
print(f"Dimensiones: {n_dimensiones}D")
print(f"Forma: {forma}")
print(f"  → {forma[0]} estaciones")
print(f"  → {forma[1]} días")
print(f"  → {forma[2]} horas por día")
print(f"Total de mediciones: {total_elementos:,}")
print(f"Tipo de datos: {tipo_datos}")
print(f"Memoria: {memoria_bytes:,} bytes ({memoria_bytes/1024:.2f} KB)")

# Ejercicio 1.1: Inspección de las propiedades del array 'temperatura'
print("--- Propiedades del array de Temperatura ---")
print(f"Dimensiones (shape): {temperatura.shape}")
print(f"Número de dimensiones (ndim): {temperatura.ndim}")
print(f"Tipo de datos (dtype): {temperatura.dtype}")
print(f"Tamaño total (elementos): {temperatura.size}")

# Extra: Inspección de los valores faltantes (NaN)
total_nan = np.isnan(temperatura).sum()
print(f"Total de valores faltantes (NaN) detectados: {total_nan}")

📊 PROPIEDADES DEL ARRAY TEMPERATURA
────────────────────────────────────────
Dimensiones: 3D
Forma: (5, 7, 24)
  → 5 estaciones
  → 7 días
  → 24 horas por día
Total de mediciones: 840
Tipo de datos: float64
Memoria: 6,720 bytes (6.56 KB)
--- Propiedades del array de Temperatura ---
Dimensiones (shape): (5, 7, 24)
Número de dimensiones (ndim): 3
Tipo de datos (dtype): float64
Tamaño total (elementos): 840
Total de valores faltantes (NaN) detectados: 7


### Ejercicio 1.2: Indexación Básica (10 puntos)

Usa indexación para extraer datos específicos.

In [6]:
# ═══════════════════════════════════════════════════════════════════
#                    EJERCICIO 1.2: INDEXACIÓN
# ═══════════════════════════════════════════════════════════════════

# TODO: Completa usando indexación

# 1. Temperatura de Coyoacán (índice 0), día 1 (índice 0), a las 12:00 (índice 12)
temp_coyoacan_d1_12h = temperatura[0, 0, 12]
print(f" Coyoacán, Día 1, 12:00h: {temp_coyoacan_d1_12h:.1f}°C")

# 2. Todas las temperaturas de Xochimilco (índice 2) en el día 3 (índice 2)
# Seleccionamos la estación 2, día 2, y todas las horas (:)
temp_xochimilco_d3 = temperatura[2, 2, :]
print(f"\n Xochimilco, Día 3 (24 horas):")
print(f"   Primeras 6 horas: {temp_xochimilco_d3[:6].round(1)}")

# 3. Temperatura promedio diario de Miguel Hidalgo (índice 4) para los 7 días
# El array temp_promedio_diario tiene shape (5, 7), así que usamos [estación, todos los días]
temp_mh_7dias = temp_promedio_diario[4, :]
print(f"\n Miguel Hidalgo - Promedio por día:")
print(f"   {temp_mh_7dias.round(1)}")

# 4. Último valor de CO2 registrado (última estación, último día, última hora)
# En Python, el índice -1 hace referencia al último elemento
ultimo_co2 = co2[-1, -1, -1]
print(f"\n Último CO2 registrado: {ultimo_co2:.1f} ppm")

 Coyoacán, Día 1, 12:00h: 27.4°C

 Xochimilco, Día 3 (24 horas):
   Primeras 6 horas: [13.9 15.4 16.2 19.3 18.9 17.8]

 Miguel Hidalgo - Promedio por día:
   [23.  22.9 22.8 23.1 23.2 23.  23.2]

 Último CO2 registrado: 401.5 ppm


### Ejercicio 1.3: Slicing Avanzado (10 puntos)

Usa slicing para extraer subconjuntos de datos.

In [8]:
# ═══════════════════════════════════════════════════════════════════
#                    EJERCICIO 1.3: SLICING
# ═══════════════════════════════════════════════════════════════════

# TODO: Completa usando slicing

# 1. Temperaturas de TODAS las estaciones, TODOS los días, horas de la TARDE (12-18)
# Nota: Si quieres incluir la hora 17 y la 18, el rango es 12:19
temp_tardes = temperatura[:, :, 12:19]
print(f" Temperaturas de tardes (12-18h)")
print(f"   Shape: {temp_tardes.shape}")

# 2. Humedad de las primeras 3 estaciones, últimos 3 días, todas las horas
# Los últimos 3 días se acceden con el índice -3:
humedad_subset = humedad[:3, -3:, :]
print(f"\n Subset de humedad")
print(f"   Shape: {humedad_subset.shape}")

# 3. CO2 de estaciones pares (0, 2, 4), todos los días, horas de mañana (6-12)
# Usamos [::2] para el paso de 2 en estaciones, y 6:12 para horas
co2_mañanas_pares = co2[::2, :, 6:12]
print(f"\n CO2 mañanas (estaciones pares)")
print(f"   Shape: {co2_mañanas_pares.shape}")

# 4. Temperaturas en orden inverso de días (del día 7 al día 1)
# El paso negativo [::-1] invierte el eje seleccionado
temp_inverso = temperatura[:, ::-1, :]
print(f"\n Temperatura días invertidos")
print(f"   Shape: {temp_inverso.shape}")

 Temperaturas de tardes (12-18h)
   Shape: (5, 7, 7)

 Subset de humedad
   Shape: (3, 3, 24)

 CO2 mañanas (estaciones pares)
   Shape: (3, 7, 6)

 Temperatura días invertidos
   Shape: (5, 7, 24)


---

## 🏋️ PARTE 2: Estadísticas Básicas (25 puntos)

### Ejercicio 2.1: Estadísticas Globales (10 puntos)

Calcula estadísticas considerando **valores NaN**.

In [9]:
# ═══════════════════════════════════════════════════════════════════
#                 EJERCICIO 2.1: ESTADÍSTICAS GLOBALES
# ═══════════════════════════════════════════════════════════════════

# IMPORTANTE: Los arrays tienen valores NaN (sensores desconectados)
# Debes usar las funciones nan* de NumPy (nanmean, nanstd, etc.)

# TODO: Calcula las siguientes estadísticas para TEMPERATURA

# 1. Temperatura promedio global
temp_promedio = np.nanmean(temperatura)

# 2. Temperatura máxima registrada
temp_maxima = np.nanmax(temperatura)

# 3. Temperatura mínima registrada
temp_minima = np.nanmin(temperatura)

# 4. Desviación estándar de temperatura
temp_std = np.nanstd(temperatura)

# 5. Rango de temperatura (máxima - mínima)
temp_rango = temp_maxima - temp_minima


print("╔══════════════════════════════════════════════════════════════╗")
print("║           ESTADÍSTICAS GLOBALES DE TEMPERATURA               ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Promedio:     {temp_promedio:>6.2f} °C                              ║")
print(f"║  Máxima:       {temp_maxima:>6.2f} °C                              ║")
print(f"║  Mínima:       {temp_minima:>6.2f} °C                              ║")
print(f"║  Desv. Est.:   {temp_std:>6.2f} °C                              ║")
print(f"║  Rango:        {temp_rango:>6.2f} °C                              ║")
print("╚══════════════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════════════╗
║           ESTADÍSTICAS GLOBALES DE TEMPERATURA               ║
╠══════════════════════════════════════════════════════════════╣
║  Promedio:      21.59 °C                              ║
║  Máxima:        32.91 °C                              ║
║  Mínima:        10.62 °C                              ║
║  Desv. Est.:     4.28 °C                              ║
║  Rango:         22.29 °C                              ║
╚══════════════════════════════════════════════════════════════╝


### Ejercicio 2.2: Estadísticas por Eje (15 puntos)

Calcula estadísticas a lo largo de ejes específicos.

```
RECORDATORIO DE EJES para array 3D (estaciones, días, horas):

       ┌─────────────────────────────────┐
      /│                                /│
     / │        axis=2 (horas)         / │
    /  │        ──────────────►       /  │
   ┌───┼─────────────────────────────┐   │
   │   │                             │   │
   │   │     axis=1 (días)           │   │
   │   │     │                       │   │
   │   │     │                       │   │
   │   │     ▼                       │   │
   │   └─────────────────────────────┼───┘
   │  /                              │  /
   │ / axis=0 (estaciones)           │ /
   │/                                │/
   └─────────────────────────────────┘

• axis=0: Opera sobre estaciones (resultado: días × horas)
• axis=1: Opera sobre días (resultado: estaciones × horas)
• axis=2: Opera sobre horas (resultado: estaciones × días)
```

In [11]:
# ═══════════════════════════════════════════════════════════════════
#                 EJERCICIO 2.2: ESTADÍSTICAS POR EJE
# ═══════════════════════════════════════════════════════════════════

# TODO: Completa los cálculos

# 1. Temperatura promedio POR ESTACIÓN 
# Promediamos sobre días (axis 1) y horas (axis 2). Nos quedan las 5 estaciones.
temp_por_estacion = np.nanmean(temperatura, axis=(1, 2))

print(" TEMPERATURA PROMEDIO POR ESTACIÓN")
print("─" * 40)
for i, est in enumerate(estaciones):
    print(f"   {est:15s}: {temp_por_estacion[i]:5.1f} °C")

# 2. Humedad promedio POR HORA DEL DÍA
# Promediamos sobre estaciones (axis 0) y días (axis 1). Nos quedan las 24 horas.
humedad_por_hora = np.nanmean(humedad, axis=(0, 1))

print("\n HUMEDAD PROMEDIO POR HORA")
print("─" * 40)
print("   Hora │ Humedad")
for h in [0, 6, 12, 18]:
    print(f"   {h:02d}:00 │ {humedad_por_hora[h]:5.1f}%")

# 3. CO2 máximo POR DÍA
# Buscamos el máximo sobre estaciones (axis 0) y horas (axis 2). Nos quedan los 7 días.
co2_max_por_dia = np.nanmax(co2, axis=(0, 2))

print("\n CO2 MÁXIMO POR DÍA")
print("─" * 40)
for d in range(n_dias):
    print(f"   Día {d+1}: {co2_max_por_dia[d]:6.1f} ppm")

 TEMPERATURA PROMEDIO POR ESTACIÓN
────────────────────────────────────────
   Coyoacán       :  21.9 °C
   Azcapotzalco   :  24.0 °C
   Xochimilco     :  20.0 °C
   Tlalpan        :  19.0 °C
   Miguel Hidalgo :  23.0 °C

 HUMEDAD PROMEDIO POR HORA
────────────────────────────────────────
   Hora │ Humedad
   00:00 │  72.1%
   06:00 │  58.2%
   12:00 │  43.7%
   18:00 │  57.1%

 CO2 MÁXIMO POR DÍA
────────────────────────────────────────
   Día 1:  471.3 ppm
   Día 2:  466.2 ppm
   Día 3:  465.0 ppm
   Día 4:  541.5 ppm
   Día 5:  466.5 ppm
   Día 6:  467.3 ppm
   Día 7:  469.4 ppm


---

## 🏋️ PARTE 3: Operaciones Vectorizadas (25 puntos)

### Ejercicio 3.1: Conversiones de Unidades (10 puntos)

Realiza conversiones usando operaciones vectorizadas (sin loops).

In [12]:
# ═══════════════════════════════════════════════════════════════════
#              EJERCICIO 3.1: CONVERSIONES VECTORIZADAS
# ═══════════════════════════════════════════════════════════════════

# TODO: Realiza las conversiones SIN usar loops (for/while)

# 1. Convertir temperatura de Celsius a Fahrenheit
# NumPy aplica la multiplicación y suma a todos los elementos del array a la vez
temperatura_fahrenheit = temperatura * 9/5 + 32

print(" TEMPERATURA EN FAHRENHEIT")
print(f"   Promedio: {np.nanmean(temperatura_fahrenheit):.1f} °F")
print(f"   Máxima:   {np.nanmax(temperatura_fahrenheit):.1f} °F")
print(f"   Mínima:   {np.nanmin(temperatura_fahrenheit):.1f} °F")

# 2. Convertir temperatura de Celsius a Kelvin
temperatura_kelvin = temperatura + 273.15

print(f"\n TEMPERATURA EN KELVIN")
print(f"   Promedio: {np.nanmean(temperatura_kelvin):.1f} K")

# 3. Normalizar humedad a rango [0, 1]
# La fórmula se aplica vectorizadamente a todo el array
humedad_min = np.nanmin(humedad)
humedad_max = np.nanmax(humedad)
humedad_normalizada = (humedad - humedad_min) / (humedad_max - humedad_min)

print(f"\n HUMEDAD NORMALIZADA [0-1]")
print(f"   Promedio: {np.nanmean(humedad_normalizada):.3f}")
print(f"   Min:      {np.nanmin(humedad_normalizada):.3f}")
print(f"   Max:      {np.nanmax(humedad_normalizada):.3f}")

 TEMPERATURA EN FAHRENHEIT
   Promedio: 70.9 °F
   Máxima:   91.2 °F
   Mínima:   51.1 °F

 TEMPERATURA EN KELVIN
   Promedio: 294.7 K

 HUMEDAD NORMALIZADA [0-1]
   Promedio: 0.510
   Min:      0.000
   Max:      1.000


### Ejercicio 3.2: Índice de Confort Térmico (15 puntos)

Calcula un índice simplificado de confort térmico combinando temperatura y humedad.

```
ÍNDICE DE CONFORT TÉRMICO (ICT)
═══════════════════════════════════════════════

Fórmula simplificada:
ICT = T + 0.05 × H

Donde:
  T = Temperatura (°C)
  H = Humedad relativa (%)

Interpretación:
  ICT < 20     → Frío
  20 ≤ ICT < 25 → Confortable
  25 ≤ ICT < 30 → Cálido
  ICT ≥ 30     → Muy caluroso
═══════════════════════════════════════════════
```

In [14]:
# ═══════════════════════════════════════════════════════════════════
#              EJERCICIO 3.2: ÍNDICE DE CONFORT TÉRMICO
# ═══════════════════════════════════════════════════════════════════

# TODO: Calcula el ICT usando broadcasting/operaciones vectorizadas

# 1. Calcular el Índice de Confort Térmico (ICT)
# NumPy realiza la operación elemento a elemento de forma vectorizada
ict = temperatura + (0.05 * humedad)

print(" ÍNDICE DE CONFORT TÉRMICO (ICT)")
print("─" * 45)
print(f"   Shape del array ICT: {ict.shape}")
print(f"   ICT promedio: {np.nanmean(ict):.2f}")
print(f"   ICT máximo:   {np.nanmax(ict):.2f}")
print(f"   ICT mínimo:   {np.nanmin(ict):.2f}")

# 2. Clasificar las condiciones usando indexación booleana
# Usamos & para combinar condiciones. Es obligatorio usar paréntesis (condición) & (condición)

n_frio = np.sum(ict < 20)
n_confortable = np.sum((ict >= 20) & (ict < 25))
n_calido = np.sum((ict >= 25) & (ict < 30))
n_muy_caluroso = np.sum(ict >= 30)

# Total de mediciones válidas (excluyendo NaN)
n_validas = np.sum(~np.isnan(ict))

print("\n DISTRIBUCIÓN DE CONDICIONES")
print("─" * 45)
print(f"    Frío (<20):           {n_frio:5d} ({100*n_frio/n_validas:5.1f}%)")
print(f"    Confortable (20-25):  {n_confortable:5d} ({100*n_confortable/n_validas:5.1f}%)")
print(f"    Cálido (25-30):       {n_calido:5d} ({100*n_calido/n_validas:5.1f}%)")
print(f"    Muy caluroso (≥30):   {n_muy_caluroso:5d} ({100*n_muy_caluroso/n_validas:5.1f}%)")
print(f"   ────────────────────────────────────────")
print(f"   Total válidas:           {n_validas:5d}")

 ÍNDICE DE CONFORT TÉRMICO (ICT)
─────────────────────────────────────────────
   Shape del array ICT: (5, 7, 24)
   ICT promedio: 24.45
   ICT máximo:   34.54
   ICT mínimo:   14.17

 DISTRIBUCIÓN DE CONDICIONES
─────────────────────────────────────────────
    Frío (<20):             103 ( 12.4%)
    Confortable (20-25):    353 ( 42.5%)
    Cálido (25-30):         325 ( 39.2%)
    Muy caluroso (≥30):      49 (  5.9%)
   ────────────────────────────────────────
   Total válidas:             830


---

## 🏋️ PARTE 4: Análisis Avanzado (25 puntos)

### Ejercicio 4.1: Detección de Anomalías (10 puntos)

Identifica valores anómalos usando el método de desviación estándar.

In [15]:
# ═══════════════════════════════════════════════════════════════════
#              EJERCICIO 4.1: DETECCIÓN DE ANOMALÍAS
# ═══════════════════════════════════════════════════════════════════

# Criterio: Un valor es anómalo si está a más de 2 desviaciones estándar
# de la media

# TODO: Detecta anomalías en CO2

# 1. Calcula la media y desviación estándar del CO2
# Usamos nanmean y nanstd para ignorar los valores faltantes (NaN)
co2_media = np.nanmean(co2)
co2_std = np.nanstd(co2)

# 2. Define los límites (media ± 2*desviaciones estándar)
limite_inferior = co2_media - (2 * co2_std)
limite_superior = co2_media + (2 * co2_std)

print(" ANÁLISIS DE ANOMALÍAS EN CO2")
print("─" * 45)
print(f"   Media CO2:      {co2_media:.1f} ppm")
print(f"   Desv. Est.:     {co2_std:.1f} ppm")
print(f"   Límite inferior: {limite_inferior:.1f} ppm")
print(f"   Límite superior: {limite_superior:.1f} ppm")

# 3. Crea una máscara booleana para valores anómalos
# Un valor es anómalo si es < inf O > sup. 
# Además, aseguramos que NO sean NaN usando ~np.isnan(co2)
mascara_anomalias = ((co2 < limite_inferior) | (co2 > limite_superior)) & (~np.isnan(co2))

# 4. Cuenta el número de anomalías
n_anomalias = np.sum(mascara_anomalias)

# 5. Obtén los valores anómalos (esto aplana el array y extrae solo los que cumplen la máscara)
valores_anomalos = co2[mascara_anomalias]

print(f"\n  ANOMALÍAS DETECTADAS: {n_anomalias}")
if n_anomalias > 0:
    print(f"   Valores: {valores_anomalos[:10].round(1)}")
    if n_anomalias > 10:
        print(f"   ... y {n_anomalias - 10} más")

 ANÁLISIS DE ANOMALÍAS EN CO2
─────────────────────────────────────────────
   Media CO2:      402.6 ppm
   Desv. Est.:     39.6 ppm
   Límite inferior: 323.4 ppm
   Límite superior: 481.7 ppm

  ANOMALÍAS DETECTADAS: 31
   Valores: [486.6 490.4 489.3 483.5 502.5 491.6 485.8 506.9 514.2 530.2]
   ... y 21 más


### Ejercicio 4.2: Análisis de Contingencia Ambiental (15 puntos)

El día 4 (índice 3) hubo una contingencia ambiental. Analiza cómo afectó las mediciones.

In [21]:
# ═══════════════════════════════════════════════════════════════════
#           EJERCICIO 4.2: ANÁLISIS DE CONTINGENCIA AMBIENTAL
# ═══════════════════════════════════════════════════════════════════

# El día 4 (índice 3) hubo contingencia ambiental
DIA_CONTINGENCIA = 3

# TODO: Completa el análisis

# 1. Extrae los datos de CO2 del día de contingencia
# Seleccionamos todas las estaciones, el día 3, y todas las horas
co2_contingencia = co2[:, DIA_CONTINGENCIA, :]

# 2. Extrae los datos de CO2 de los días normales (ya proporcionado)
dias_normales = [0, 1, 2, 4, 5, 6]
co2_dias_normales = co2[:, dias_normales, :]

# 3. Calcula el promedio de CO2 en el día de contingencia
promedio_contingencia = np.nanmean(co2_contingencia)

# 4. Calcula el promedio de CO2 en días normales
promedio_normal = np.nanmean(co2_dias_normales)

# 5. Calcula el incremento porcentual
incremento_porcentual = ((promedio_contingencia - promedio_normal) / promedio_normal) * 100

print("╔══════════════════════════════════════════════════════════════╗")
print("║            ANÁLISIS DE CONTINGENCIA AMBIENTAL                ║")
print("║                          Día 4                               ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  CO2 promedio día contingencia: {promedio_contingencia:>7.1f} ppm                  ║")
print(f"║  CO2 promedio días normales:    {promedio_normal:>7.1f} ppm                  ║")
print(f"║  Incremento:                    {incremento_porcentual:>7.1f} %                    ║")
print("╚══════════════════════════════════════════════════════════════╝")

# 6. Identifica la estación más afectada
# Promediamos sobre el eje de horas (axis=1) para contingencia (que tiene shape 5,24)
co2_por_estacion_contingencia = np.nanmean(co2_contingencia, axis=1)

# Promediamos sobre días y horas (axis=(1,2)) para días normales (que tiene shape 5,6,24)
co2_por_estacion_normal = np.nanmean(co2_dias_normales, axis=(1, 2))

# Incremento por estación
incremento_por_estacion = ((co2_por_estacion_contingencia - co2_por_estacion_normal) / 
                           co2_por_estacion_normal) * 100

# Índice de la estación más afectada usando argmax (devuelve el índice del valor más alto)
idx_mas_afectada = np.argmax(incremento_por_estacion)

print("\n IMPACTO POR ESTACIÓN")
print("─" * 50)
for i, est in enumerate(estaciones):
    barra = "█" * int(incremento_por_estacion[i] / 2)
    print(f"   {est:15s}: +{incremento_por_estacion[i]:5.1f}% {barra}")

print(f"\n  Estación más afectada: {estaciones[idx_mas_afectada]}")

╔══════════════════════════════════════════════════════════════╗
║            ANÁLISIS DE CONTINGENCIA AMBIENTAL                ║
║                          Día 4                               ║
╠══════════════════════════════════════════════════════════════╣
║  CO2 promedio día contingencia:   453.4 ppm                  ║
║  CO2 promedio días normales:      394.1 ppm                  ║
║  Incremento:                       15.1 %                    ║
╚══════════════════════════════════════════════════════════════╝

 IMPACTO POR ESTACIÓN
──────────────────────────────────────────────────
   Coyoacán       : + 15.7% ███████
   Azcapotzalco   : + 15.0% ███████
   Xochimilco     : + 15.2% ███████
   Tlalpan        : + 14.3% ███████
   Miguel Hidalgo : + 15.2% ███████

  Estación más afectada: Coyoacán


---

## 🎯 EJERCICIO FINAL: Reporte Ejecutivo (BONUS - 10 puntos)

Genera un reporte resumido con las métricas más importantes.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    EJERCICIO BONUS: REPORTE EJECUTIVO
# ═══════════════════════════════════════════════════════════════════

# TODO: Completa las métricas del reporte

# 1. Estación más calurosa (mayor temperatura promedio)
idx_mas_calurosa = ___
estacion_mas_calurosa = estaciones[idx_mas_calurosa]

# 2. Estación más húmeda (mayor humedad promedio)
idx_mas_humeda = ___
estacion_mas_humeda = estaciones[idx_mas_humeda]

# 3. Estación con mejor calidad de aire (menor CO2 promedio)
idx_mejor_aire = ___
estacion_mejor_aire = estaciones[idx_mejor_aire]

# 4. Hora más calurosa del día (promedio de todas las estaciones y días)
temp_por_hora = np.nanmean(temperatura, axis=(0, 1))
hora_mas_calurosa = ___

# 5. Hora con peor calidad de aire
co2_por_hora = np.nanmean(co2, axis=(0, 1))
hora_peor_aire = ___

# 6. Número de valores faltantes en total
nan_temperatura = ___
nan_humedad = ___
nan_co2 = ___
total_nan = nan_temperatura + nan_humedad + nan_co2

print("")
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║                                                                      ║")
print("║            🌡️  METEOSENSE - REPORTE EJECUTIVO SEMANAL  💨            ║")
print("║                        CDMX - Semana de Análisis                     ║")
print("║                                                                      ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║                                                                      ║")
print("║  📊 RESUMEN DE CONDICIONES                                           ║")
print("║  ─────────────────────────────────────────────────────────────────   ║")
print(f"║    🌡️  Temperatura promedio:    {np.nanmean(temperatura):>5.1f} °C                        ║")
print(f"║    💧 Humedad promedio:         {np.nanmean(humedad):>5.1f} %                         ║")
print(f"║    🏭 CO2 promedio:            {np.nanmean(co2):>6.1f} ppm                       ║")
print("║                                                                      ║")
print("║  🏆 RANKINGS                                                         ║")
print("║  ─────────────────────────────────────────────────────────────────   ║")
print(f"║    🔥 Estación más calurosa:   {estacion_mas_calurosa:15s}                  ║")
print(f"║    💧 Estación más húmeda:     {estacion_mas_humeda:15s}                  ║")
print(f"║    🌿 Mejor calidad de aire:   {estacion_mejor_aire:15s}                  ║")
print("║                                                                      ║")
print("║  ⏰ PATRONES TEMPORALES                                              ║")
print("║  ─────────────────────────────────────────────────────────────────   ║")
print(f"║    🌡️  Hora más calurosa:       {hora_mas_calurosa:02d}:00 hrs                          ║")
print(f"║    🏭 Hora con más CO2:         {hora_peor_aire:02d}:00 hrs                          ║")
print("║                                                                      ║")
print("║  ⚠️  CALIDAD DE DATOS                                                ║")
print("║  ─────────────────────────────────────────────────────────────────   ║")
print(f"║    Valores faltantes totales:  {total_nan:4d}                                 ║")
print(f"║      - Temperatura: {nan_temperatura:3d}                                            ║")
print(f"║      - Humedad:     {nan_humedad:3d}                                            ║")
print(f"║      - CO2:         {nan_co2:3d}                                            ║")
print("║                                                                      ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
print("")

---

## ✅ Checklist de Entrega

Antes de entregar, verifica que:

| # | Criterio | Puntos |
|---|----------|--------|
| 1 | Parte 1: Exploración de Arrays completada | 25 |
| 2 | Parte 2: Estadísticas Básicas completadas | 25 |
| 3 | Parte 3: Operaciones Vectorizadas completadas | 25 |
| 4 | Parte 4: Análisis Avanzado completado | 25 |
| 5 | BONUS: Reporte Ejecutivo completado | +10 |
| | **Total posible** | **110** |

### 📝 Notas Importantes

- **NO uses loops** (for, while) en las partes 2 y 3 - usa operaciones vectorizadas
- **Usa funciones nan*** cuando trabajes con datos que tienen valores faltantes
- **Documenta tu código** si haces algo no obvio
- El código debe ejecutarse sin errores de principio a fin

---

### 🏅 Criterios de Evaluación

```
RÚBRICA DE CALIFICACIÓN
═══════════════════════════════════════════════════════

✓ Código correcto y funcional         → 70%
✓ Uso apropiado de NumPy              → 15%
  (vectorización, funciones nan, ejes)
✓ Resultados razonables               → 10%
✓ Código limpio y organizado          → 5%

═══════════════════════════════════════════════════════
```

---

*MeteoSense Analytics - Reto desarrollado para Programación para Ciencia de Datos, IPN 2026*